# 02 — Baseline модели (univariate)
**Цель**: установить нижнюю планку качества.
Показать, что модели на лагах целевой переменной просто «копируют последнее значение».

Модели: Naive | Moving Average | ARIMA | XGBoost univariate

## 0. Зависимости

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from warnings import filterwarnings
filterwarnings("ignore")
from statsmodels.tsa.arima.model import ARIMA
from xgboost import XGBRegressor
from utils import (compute_metrics, metrics_table, smooth_series, mark_omicron,
                   train_test_split_temporal, add_lag_features,
                   plot_forecast, plot_metrics_bar, set_plot_style)
set_plot_style()
FIGURES = "../results/figures"
METRICS = "../results/metrics"
os.makedirs(FIGURES, exist_ok=True)
os.makedirs(METRICS, exist_ok=True)
TEST_DAYS = 31
print("Зависимости загружены ✓")

## 1. Загрузка данных

In [ ]:
# ─── Реальная загрузка ───
# from utils import load_covid_data
# df = load_covid_data("../data/raw/covid_spb.csv")

# ─── Заглушка ───
date_range = pd.date_range("2021-01-01", "2023-06-30", freq="D")
np.random.seed(42)
n = len(date_range)
trend = np.concatenate([np.exp(np.linspace(4,6,365)),
                        np.exp(np.linspace(6,8,365))*3,
                        np.exp(np.linspace(8,5,n-730))])
noise = np.random.normal(0, trend*0.15)
cases = np.maximum(trend+noise, 0).astype(int)
df = pd.DataFrame({"date": date_range, "new_cases": cases})
df["new_cases_smooth"] = smooth_series(df["new_cases"])
df = mark_omicron(df)
TARGET = "new_cases_smooth"
train_df, test_df = train_test_split_temporal(df, TEST_DAYS)
y_train = train_df[TARGET].values
y_test  = test_df[TARGET].values
dates_test = test_df["date"].values
print(f"Train: {len(train_df)} | Test: {len(test_df)}")

## 2. Naive baseline

In [ ]:
naive_pred = np.full(len(y_test), y_train[-1])
m_naive = compute_metrics(y_test, naive_pred, "Naive (last value)")
plot_forecast(dates_test, y_test, naive_pred, "Naive",
              save_path=f"{FIGURES}/02_naive.png")
plt.show()
print(m_naive.to_string(index=False))

## 3. Moving Average (MA-7)

In [ ]:
ma_pred = np.full(len(y_test), np.mean(y_train[-7:]))
m_ma = compute_metrics(y_test, ma_pred, "Moving Average (7d)")
print(m_ma.to_string(index=False))

## 4. ARIMA(7,1,2)

In [ ]:
arima_model = ARIMA(y_train, order=(7,1,2))
arima_fit   = arima_model.fit()
arima_pred  = arima_fit.forecast(steps=TEST_DAYS)
m_arima = compute_metrics(y_test, arima_pred, "ARIMA(7,1,2)")
plot_forecast(dates_test, y_test, arima_pred, "ARIMA(7,1,2)",
              save_path=f"{FIGURES}/02_arima.png")
plt.show()
print(m_arima.to_string(index=False))

## 5. XGBoost Univariate (лаги 1–14)

In [ ]:
LAG_COLS = [f"{TARGET}_lag{i}" for i in range(1,15)]
df_lag = add_lag_features(df, TARGET, lags=range(1,15)).dropna()
train_lag = df_lag.iloc[:-TEST_DAYS]
test_lag  = df_lag.iloc[-TEST_DAYS:]
xgb_uni = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=5,
                        subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0)
xgb_uni.fit(train_lag[LAG_COLS].values, train_lag[TARGET].values)
xgb_uni_pred = xgb_uni.predict(test_lag[LAG_COLS].values)
m_xgb_uni = compute_metrics(test_lag[TARGET].values, xgb_uni_pred, "XGBoost Univariate")
plot_forecast(test_lag["date"].values, test_lag[TARGET].values, xgb_uni_pred,
              "XGBoost Univariate", save_path=f"{FIGURES}/02_xgb_univariate.png")
plt.show()
print(m_xgb_uni.to_string(index=False))

## 6. Итоговая таблица

In [ ]:
all_metrics = metrics_table([m_naive, m_ma, m_arima, m_xgb_uni])
all_metrics.to_csv(f"{METRICS}/02_baseline_metrics.csv", index=False)
print("=== BASELINE СРАВНЕНИЕ ===")
print(all_metrics.to_string(index=False))
plot_metrics_bar(all_metrics, "RMSE", save_path=f"{FIGURES}/02_baseline_rmse.png")
plt.show()

## 7. Вывод
> XGBoost Univariate и ARIMA дают схожие метрики с Naive baseline.
> Модели на лагах целевой переменной копируют последнее значение — без реальной прогностической силы.
> Это мотивирует переход к мультивариантной постановке.

**Следующий шаг** → `03_multivariate.ipynb`